# LLM Latency Benchmark

Mäter hur svarstid varierar med `max_new_tokens`.  
Fast prompt, 5 körningar per konfiguration → medelvärde och spridning.

**Syfte:** hitta optimal token-längd givet acceptabel latens för live-användning på golfbanan.

In [ ]:
import time
import pandas as pd
import matplotlib.pyplot as plt
from transformers import pipeline

print('Laddar modell (cachas efter första körning)...')
llm = pipeline('text-generation', model='HuggingFaceTB/SmolLM2-135M-Instruct')
print('Klar.')

In [ ]:
# Fast prompt — isolerar max_new_tokens som enda variabel
PROMPT = [
    {
        'role': 'user',
        'content': (
            'Spelarens stats: GIR 38.9% (PGA 65%), '
            'Fairway 50% (PGA 60%), Putts/hal 2.28 (PGA 1.73). '
            'Vad ar min svagaste del?'
        )
    }
]

TOKEN_LENGTHS = list(range(30, 131, 10))
RUNS_PER_LENGTH = 5

results = []

for max_tokens in TOKEN_LENGTHS:
    print(f'max_new_tokens={max_tokens} - kor {RUNS_PER_LENGTH}x ...', end=' ')
    for run in range(RUNS_PER_LENGTH):
        t0 = time.perf_counter()
        out = llm(PROMPT, max_new_tokens=max_tokens)
        elapsed = time.perf_counter() - t0

        raw_text = out[0]['generated_text'][-1]['content']
        actual_tokens = len(raw_text.split())

        results.append({
            'max_new_tokens': max_tokens,
            'run': run + 1,
            'elapsed_s': round(elapsed, 3),
            'approx_tokens_generated': actual_tokens,
            'tokens_per_sec': round(actual_tokens / elapsed, 1),
            'raw_text': raw_text,
        })
    print('klar')

df = pd.DataFrame(results)
df[['max_new_tokens', 'run', 'elapsed_s', 'approx_tokens_generated', 'tokens_per_sec']]

In [ ]:
# Aggregera per token-längd
summary = (
    df.groupby('max_new_tokens')
    .agg(
        mean_s=('elapsed_s', 'mean'),
        std_s=('elapsed_s', 'std'),
        min_s=('elapsed_s', 'min'),
        max_s=('elapsed_s', 'max'),
        mean_tokens=('approx_tokens_generated', 'mean'),
    )
    .round(3)
    .reset_index()
)
summary

In [ ]:
# Kvalitetsgranskning — ett representativt svar per token-längd (run 3 = varken första eller sista)
print("=== KVALITET PER TOKEN-LÄNGD ===\n")
for max_tokens in TOKEN_LENGTHS:
    sample = df[(df['max_new_tokens'] == max_tokens) & (df['run'] == 3)].iloc[0]
    print(f"--- max_new_tokens={max_tokens} ({sample['approx_tokens_generated']} tokens, {sample['elapsed_s']}s) ---")
    print(sample['raw_text'].strip())
    print()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Vänster: svarstid med felstaplar
axes[0].errorbar(
    summary['max_new_tokens'],
    summary['mean_s'],
    yerr=summary['std_s'],
    marker='o', capsize=5, linewidth=2
)
axes[0].set_xlabel('max_new_tokens')
axes[0].set_ylabel('Svarstid (s)')
axes[0].set_title('Latens vs token-längd (medel ± std)')
axes[0].grid(True, alpha=0.3)

# Höger: alla individuella körningar
for tokens in TOKEN_LENGTHS:
    subset = df[df['max_new_tokens'] == tokens]
    axes[1].scatter(
        [tokens] * len(subset),
        subset['elapsed_s'],
        alpha=0.6, s=40
    )
axes[1].set_xlabel('max_new_tokens')
axes[1].set_ylabel('Svarstid (s)')
axes[1].set_title('Individuella körningar')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Experiment 1 — Batch inference

Skickar N prompts i en enda `pipeline()`-körning med `batch_size=N` och mäter genomströmning (sekunder per prompt).

**Hypotes:** transformers internt optimerar tokenization och kan ge bättre throughput än N sekventiella anrop.  
**Relevant för:** post-runda batch-parsing av 18 hål-memon.

In [ ]:
import time
import pandas as pd

BATCH_PROMPT_CONTENT = (
    'Spelarens stats: GIR 38.9% (PGA 65%), '
    'Fairway 50% (PGA 60%), Putts/hal 2.28 (PGA 1.73). '
    'Vad ar min svagaste del?'
)

BATCH_SIZES = [5, 10, 15]
BATCH_MAX_TOKENS = 60  # fast — isolerar batch-effekten

batch_results = []

for n in BATCH_SIZES:
    prompts = [[{'role': 'user', 'content': BATCH_PROMPT_CONTENT}]] * n
    print(f'Batch size={n} ...', end=' ')
    t0 = time.perf_counter()
    outputs = llm(prompts, max_new_tokens=BATCH_MAX_TOKENS, batch_size=n)
    total_s = time.perf_counter() - t0
    per_prompt_s = total_s / n
    print(f'{total_s:.2f}s total, {per_prompt_s:.3f}s/prompt')
    batch_results.append({
        'batch_size': n,
        'total_s': round(total_s, 3),
        'per_prompt_s': round(per_prompt_s, 3),
    })

# Baseline: sekventiell tid med samma token-längd (from summary above)
baseline_per_prompt = summary.loc[summary['max_new_tokens'] == BATCH_MAX_TOKENS, 'mean_s'].values[0]

df_batch = pd.DataFrame(batch_results)
df_batch['baseline_s'] = baseline_per_prompt
df_batch['speedup'] = (df_batch['baseline_s'] / df_batch['per_prompt_s']).round(2)
print()
print(df_batch.to_string(index=False))

## Experiment 2 — Async parallella anrop

Skickar N async anrop mot den lokala modellen och mäter total tid.

**Hypotes:** serialiseras av Python GIL och CPU-begränsning — ingen reell vinst jämfört med sekventiellt.  
**Syfte:** bekräfta eller motbevisa om async hjälper för lokal modell.

In [ ]:
import asyncio
import concurrent.futures
import time
import pandas as pd

ASYNC_MAX_TOKENS = 60  # matchar Experiment 1 för jämförbarhet

def run_single(prompt_content: str) -> float:
    """Kör ett pipeline-anrop och returnerar elapsed tid."""
    t0 = time.perf_counter()
    llm(
        [{'role': 'user', 'content': prompt_content}],
        max_new_tokens=ASYNC_MAX_TOKENS,
    )
    return time.perf_counter() - t0


async def run_parallel(n: int) -> float:
    """Kör n anrop parallellt i en thread-pool och returnerar total wall-clock tid."""
    loop = asyncio.get_running_loop()
    with concurrent.futures.ThreadPoolExecutor(max_workers=n) as pool:
        t0 = time.perf_counter()
        tasks = [
            loop.run_in_executor(pool, run_single, BATCH_PROMPT_CONTENT)
            for _ in range(n)
        ]
        await asyncio.gather(*tasks)
        return time.perf_counter() - t0


ASYNC_SIZES = [5, 10, 15]
async_results = []

for n in ASYNC_SIZES:
    print(f'Async n={n} ...', end=' ')
    total_s = await run_parallel(n)
    per_prompt_s = total_s / n
    print(f'{total_s:.2f}s total, {per_prompt_s:.3f}s/prompt')
    async_results.append({
        'n_concurrent': n,
        'total_s': round(total_s, 3),
        'per_prompt_s': round(per_prompt_s, 3),
    })

df_async = pd.DataFrame(async_results)
df_async['baseline_s'] = baseline_per_prompt
df_async['speedup'] = (df_async['baseline_s'] / df_async['per_prompt_s']).round(2)
print()
print(df_async.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))

ax.axhline(baseline_per_prompt, color='gray', linestyle='--', label=f'Sekventiell baseline ({baseline_per_prompt:.2f}s/prompt)')
ax.plot(df_batch['batch_size'], df_batch['per_prompt_s'], marker='o', label='Exp 1: Batch inference')
ax.plot(df_async['n_concurrent'], df_async['per_prompt_s'], marker='s', label='Exp 2: Async parallellt')

ax.set_xlabel('Antal prompts')
ax.set_ylabel('Tid per prompt (s)')
ax.set_title('Batch vs Async vs Sekventiell (max_new_tokens=60)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n=== SLUTSATS ===")
print(f"Batch speedup vid n=15:  {df_batch.loc[df_batch['batch_size']==15, 'speedup'].values[0]}x")
print(f"Async speedup vid n=15:  {df_async.loc[df_async['n_concurrent']==15, 'speedup'].values[0]}x")